# 认识消息
大模型没有记忆，只能根据当前消息(上下文)进行响应。
很多场景下，我们需要根据历史消息(上下文)进行响应。
如果应用需要记住历史消息(上下文)，则需要再程序中维护消息列表

在langchian中，Message(消息)是模型交互的最基本单元。
代表输入或输出（input/output）

## 消息的内部结构
Role: 角色（user/assistant/system）
- 系统消息：系统提示词，用于对话开始前为模型设定角色、行为准则和上下文背景它像是给AI助手的一份工作说明书，决定了其回答问题的风格、领域和专业范围。
{"role": "system", "content": "你是个精通编程的软件架构师"}
- 用户信息：用户提示词，用于向模型发送问题或指令。
{"role": "user", "content": "你好"}
- 助手信息：模型的响应，包含问题的答案或指令的执行。
{"role": "assistant", "content": "你好，有什么我可以帮助你的吗？"}
- 工具调用的信息：模型调用外部工具或函数，用于执行更复杂的任务。
{"role": "tool", "content": "今天天气很好", "tool_call_id":
"call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"}


问题：为什么使用不同的消息类型？

明确角色 ：清晰区分系统提示、用户输入和 AI 回复

控制行为 ：通过 SystemMessage 精确控制 AI 的行为

对话历史 ：构建完整的多轮对话上下文

调试友好 ：更容易追踪和调试对话流程


Content: 消息内容

Metadata: （可选的）元数据
消息的元数据可以包含消息的发送时间、发送者、接收者等信息。

## 格式
- json

- 对象格式

举例1：JSON格式

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()
import os

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

# 初始化格式
messages = [
    {"role": "system", "content": "你是个精通编程的软件架构师"},
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "你好，有什么我可以帮助你的吗？"},
    {"role": "user", "content": "你好"}
]

response = model.invoke(messages)
print(response.content)


ValidationError: 1 validation error for ChatDeepSeek
api_base
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type

举例2：对象格式

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage 

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE = os.getenv("DEEPSEEK_API_BASE")

model = init_chat_model(
    model_name="deepseek:deepseek-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_API_BASE
)

messages1= [
    SystemMessage(content="你是个精通编程的软件架构师"),
    HumanMessage(content="你好"),
    AIMessage(content="你好，有什么我可以帮助你的吗？"),
    HumanMessage(content="你好"),
]


# 或者
messages = [
    SystemMessage("你是个精通编程的软件架构师"),
    HumanMessage("你好"),
    AIMessage("你好，有什么我可以帮助你的吗？"),
    HumanMessage("你好"),
]

response = model.invoke(messages)
print(response.content)


## 消息对象字段说明
content ：消息内容，字段名可以省略

SystemMessage参数列表相当于SystemMessage(content="你是个精通编程的软件架构师")

HumanMessage参数列表相当于HumanMessage(content="你好")

注意这里可以带元数据字段metadata

HumanMessage(

    content="Hello!",

    name="alice", # 可选，用户名

    id="msg_123", # 可选，message的ID

    metadata={"user_id": "123456"}, # 可选，自定义元数据

)

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek:deepseek-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

messages = [
    SystemMessage("你是一个信息提取器。你会收到多条来自不同发言者的 user 消息。每条消息可能带有 name 字段。你的任务是：严格根据每条消息的 name 提取发言者及其观点，并输出 JSON。禁止使用“第一个人/第二个人”这种相对称呼。若某条消息没有 name，则输出 unknown。输出格式：{\"speakers\": [{\"name\":\"...\", \"claim\":\"...\"}]}"),
    HumanMessage(
        content="我认为 1+1=2",
        name="Bob"
    ),
    HumanMessage(
        content="我认为 1+1>2",
        name="Tom"
    ),
    HumanMessage(
        content="请列出谁说了什么，不要判断对错。",
        name="audience"
    )
]

response = model.invoke(messages)
print(response.content)

模型加载了name传递的消息，这在多人对话中非常有用。
拓展：使用ChatOpenRouter没有name字段

AIMessage参数列表

同样可以省略

In [ ]:
from langchain_core.messages import SystemMessage,HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from rich import print as rprint

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 2、模型的初始化
model = init_chat_model(
    model = "deepseek-flash",
    model_provider = "deepseek",
    api_key = DEEPSEEK_API_KEY,
    api_base = DEEPSEEK_BASE_URL,
)

messages = [
    SystemMessage("你叫小智，是一名助人为乐的助手。"),
    HumanMessage("你好，好久不见，请介绍下你自己。")
]

response = model.invoke(messages)
rprint(response)

## ToolMessage的使用

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from rich import print as rprint

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    api_base=os.getenv("DEEPSEEK_BASE_URL"),
)

def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    return "阴雨连绵"

model_with_tools = model.bind_tools([get_weather])

# ① 让模型自己产生 tool_calls
messages = [HumanMessage(content="今天北京天气如何")]
ai_message = model_with_tools.invoke(messages)
rprint("模型返回的 tool_calls:", ai_message.tool_calls)
messages.append(ai_message)

# ② 用模型给的 id 回传工具结果
for tc in ai_message.tool_calls:
    result = get_weather(**tc["args"])
    messages.append(ToolMessage(content=result, tool_call_id=tc["id"]))

# ③ 第二次调用，拿到自然语言回答
final = model_with_tools.invoke(messages)
rprint("最终回复:", final.content)

# 对话历史的优化

In [ ]:
def keep_recent_messages(messages, max_pairs: 3):
    """
    保留最近的N轮对话
    max_pairs: 保留对话的轮数（每轮 = user + assistant)
    """

    # 分离system 消息和对话消息
    system_messages = [m for m in messages if m.get("role") == "system"]
    conversation_messages = [m for m in messages if m.get("role") != "system"]

    # 只保留最近的消息对
    recent_messages = conversation_messages[-(max_pairs * 2):]

    # 返回最近的消息对
    return system_messages + recent_messages

In [ ]:
# 初始化
long_conversation = [
    {"role": "system", "content": "你是 Python 导师"}
]

# 第 1 轮
long_conversation.append({"role": "user", "content": "什么是列表？用一句解释"})
r1 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r1.content})

# 第 2 轮
long_conversation.append({"role": "user", "content": "列表和元组有什么区别？用一句解释"})
r2 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r2.content})

# 第 3 轮
long_conversation.append({"role": "user", "content": "什么是字典呢？用一句解释"})
r3 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r3.content})

print(f"原始消息数:{len(long_conversation)}")

# 优化：只保留最近2轮
optimized = keep_recent_messages(long_conversation, max_pairs = 2)

print(f"优化后消息: {len(optimized)}")
print(f"保留的内容:system + 最近2轮对话")

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override = True)

MODEL_NAME = "deepseek-flash"
MAX_PAIRS_HISTORY = 10
EXIT_WORD = "quit"

model = init_chat_model(
    model = MODEL_NAME,
    model_provider = "deepseek",
    api_key = os.getenv("DEEPSEEK_API_KEY"),
    base_url = os.getenv("DEEPSEEK_BASE_URL"),
)

# 维护一个消息列表
messages = [
    {
        "role": "system",
        "content":"你是小谷，是江来教育的数字员工，也是一名可爱的AI助手"
    }
]

print(f"请输入具体的问题，当输入{EXIT_WORD}的时候就结束对话")

i = 1 # 描述对话的轮数
while True:
    print("\n", "=" * 10, f"第{i}轮对话开始", "=" * 10, "\n")

    user_input = input("请输入：")

    # 判断是否结束当前会话
    if user_input == EXIT_WORD:
        print("会话已结束")
        break

    # 将用户的信息添加到消息列表中
    messages.append(
        {
            "role":"user",
            "content":user_input
        }
    )

    # 拼接AI回复的消息信息
    reply_content = ""

    print("小谷:", end = "", flush = True)

    # 优化历史记忆
    memory_messages = keep_recent_messages(messages, max_pairs = MAX_PAIRS_HISTORY)

    for chunk in model.stream(memory_messages):
        if chunk.content:
            print(chunk.content, end = "", flush = True)
            reply_content += chunk.content

    print("\n", "=" * 10, f"第{i}轮对话结束", "=" * 10, "\n")

    i += 1

    # 将模型的响应添加到消息列表
    messages.append({"role":"assistant", "content":reply_content})